# M2-B2 — Fine-tune YOLOv8n หลังสลับ SiLU→LeakyReLU (Google Colab GPU)

**เป้าหมาย:** ให้ conv/bn weight ปรับตัวเข้ากับ LeakyReLU ผ่าน forward จริงตอน train → logit range หด (จาก ±2000) → int8 quantize ไม่ saturate → cos_sim กลับขึ้นผ่าน 0.99

**วิธีใช้:** Runtime → Change runtime type → **T4 GPU** แล้วกด Run all (Ctrl+F9)

**Output:** ไฟล์ `yolov8n_leaky_state.pt` (plain state_dict) จะถูก download อัตโนมัติตอนจบ — เอาไปวางใน `03-model/phase0-wsl-recovered/` แล้วรัน merge script ฝั่ง container ตามที่บอกไว้ในเซลล์สุดท้าย

In [ ]:
# 0) เช็คว่าได้ GPU จริง (ต้องเห็น Tesla T4 หรือรุ่นอื่น ไม่ใช่ error)
!nvidia-smi -L

In [ ]:
# 1) ติดตั้ง ultralytics (เวอร์ชันล่าสุด — เข้ากับ torch ของ Colab ได้เอง)
!pip install -q ultralytics
import torch, ultralytics
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('ultralytics', ultralytics.__version__)

In [ ]:
# 2) Fine-tune: โหลด yolov8n.pt (pretrained มาตรฐาน) → สลับ SiLU→LeakyReLU ผ่าน callback → train บน coco128
import torch, torch.nn as nn
from ultralytics import YOLO

# slope ต้องตรงกับ replace_silu_with_leakyrelu ใน quantize_yolo_pytorch.py เป๊ะ
SLOPE = 0.1015625

def swap_silu_to_leaky(module):
    """แทน SiLU ทุกตัวด้วย LeakyReLU (inplace=False กัน autograd inplace error ตอน train;
    numerically เท่ากับ inplace=True ที่ quantize ใช้)"""
    n = 0
    for name, child in module.named_children():
        if isinstance(child, nn.SiLU):
            setattr(module, name, nn.LeakyReLU(SLOPE, inplace=False)); n += 1
        else:
            n += swap_silu_to_leaky(child)
    return n

model = YOLO('yolov8n.pt')  # auto-download standard pretrained

def _cb(trainer):
    # กันบั๊ก: swap ตอน train จริง (ไม่พึ่งว่า ultralytics จะเก็บ substitution ให้)
    n = swap_silu_to_leaky(trainer.model)
    m = 0
    if getattr(trainer, 'ema', None) is not None and getattr(trainer.ema, 'ema', None) is not None:
        m = swap_silu_to_leaky(trainer.ema.ema)
    print(f'[cb] SiLU->LeakyReLU({SLOPE}) swapped: model={n} ema={m}')
    assert n > 0, 'FATAL: ไม่พบ SiLU — activation อาจไม่ใช่ SiLU'

model.add_callback('on_pretrain_routine_end', _cb)

results = model.train(
    data='coco128.yaml', epochs=50, imgsz=640, batch=16, device=0,
    optimizer='SGD', lr0=0.01, warmup_epochs=3.0,   # fine-tune ต่อจาก pretrained
    plots=False, val=True, verbose=True,
    # อย่าใส่ pretrained=False เด็ดขาด — มันโยนน้ำหนัก pretrained ทิ้งแล้ว init สุ่ม
    # (เทรนจากศูนย์) coco128 128 รูปเรียนไม่ทัน -> mAP~0 = บั๊กที่เจอรอบแรก
)
print('best.pt =', results.save_dir)

# >>> เช็คด่วน: mAP50 ตอน val แต่ละ epoch ควร "สูงตั้งแต่ต้น" (>0.2) เพราะ fine-tune
#     จาก pretrained ถ้ายังเห็น mAP~0 ทุก epoch = ผิด (เทรนจากศูนย์) อย่าเอา weights ไปใช้


In [ ]:
# 3) เซฟเป็น PLAIN state_dict (tensor ล้วน) — โหลดข้ามเวอร์ชัน ultralytics/torch ได้
#    (ห้ามเซฟ model object เพราะ container ใช้ ultralytics 8.0.x จะ unpickle ไม่ตรงเวอร์ชัน)
import os, torch
best = os.path.join(str(results.save_dir), 'weights', 'best.pt')
ckpt = torch.load(best, map_location='cpu', weights_only=False)
m = ckpt['ema'] if ckpt.get('ema') is not None else ckpt['model']
sd = m.float().state_dict()
torch.save(sd, 'yolov8n_leaky_state.pt')
print('saved yolov8n_leaky_state.pt :', len(sd), 'tensors')

# sanity: ดู logit range คร่าว ๆ ว่าหดจริงไหม (รันโมเดลกับ 1 รูป coco128)
import glob, numpy as np, cv2
m = m.float().eval()
imgp = sorted(glob.glob('/content/datasets/coco128/images/train2017/*.jpg'))[0]
im = cv2.cvtColor(cv2.resize(cv2.imread(imgp), (640, 640)), cv2.COLOR_BGR2RGB).astype('float32')/255.0
x = torch.from_numpy(np.transpose(im, (2, 0, 1))[None])
with torch.no_grad():
    out = m(x)
raw = out[1] if isinstance(out, (list, tuple)) and len(out) == 2 else out
if isinstance(raw, (list, tuple)):
    for t in raw:
        t = t.numpy(); print(f'  head {tuple(t.shape)} logit range [{t.min():.1f}, {t.max():.1f}]')
print('>>> ก่อน fine-tune เคยเห็น ±2000 ที่ 20x20 — ถ้าตอนนี้หดเหลือหลักสิบ = สำเร็จ')

In [ ]:
# 4) download กลับเครื่อง
from google.colab import files
files.download('yolov8n_leaky_state.pt')

## ขั้นต่อไป (ทำบนเครื่องตัวเอง หลังได้ `yolov8n_leaky_state.pt`)

1. วางไฟล์ `yolov8n_leaky_state.pt` ไว้ใน `03-model/phase0-wsl-recovered/`
2. บอก Claude ว่า **"ได้ weights จาก Colab แล้ว"** — Claude จะรัน (ใน Vitis AI container):
   - `merge_state_into_ckpt.py` → สร้าง `yolov8n_leaky_ft.pt` (container-native)
   - re-calib + re-verify (`verify_task2` part B) → เช็ค cos_sim ผ่าน 0.99 ทั้ง 3 tensor